In [4]:
import numpy as np
import pandas as pd
import pymc as pm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import geopandas as gpd
from pymc.gp.util import plot_gp_dist
import arviz as az
# import pygeos
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')


In [6]:

# Load the data
data_2015 = pd.read_csv('./data/exports/malaria_2015_selected.csv')

# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(data_2015['LONGNUM'], data_2015['LATNUM'])]
gdf_2015 = gpd.GeoDataFrame(data_2015, geometry=geometry, crs="EPSG:4326")

# Calculate positive malaria cases (this depends on your data structure)
# Based on the column names, I'm guessing these represent positive cases and total tested
gdf_2015['positive_cases'] = gdf_2015['total_count_shch'] * gdf_2015['MP_15']  # Approximation based on prevalence
gdf_2015['positive_cases'] = gdf_2015['positive_cases'].round().astype(int)

# Prepare the data for modeling
X = gdf_2015[['Rain15', 'Nightligh', 'DLST_15', 'EVI_15', 'ITN_15', 'prop_households_ITN']]
X = X.fillna(X.mean())  # Handle missing values
X_scaled = (X - X.mean()) / X.std()  # Scale the predictors

# Coordinates for spatial modeling
coords = np.array([gdf_2015.geometry.x, gdf_2015.geometry.y]).T

# Response variables
n_trials = gdf_2015['total_count_shch'].values
positive = gdf_2015['positive_cases'].values

In [9]:

# Calculate positive malaria cases (this depends on your data structure)
# Based on the column names, I'm guessing these represent positive cases and total tested
gdf_2015['positive_cases'] = gdf_2015['total_count_shch'] * gdf_2015['MP_15']  # Approximation based on prevalence
gdf_2015['positive_cases'] = gdf_2015['positive_cases'].round().astype(int)

# Prepare the data for modeling
X = gdf_2015[['Rain15', 'Nightligh', 'DLST_15', 'EVI_15', 'ITN_15', 'prop_households_ITN']]
X = X.fillna(X.mean())  # Handle missing values
X_scaled = (X - X.mean()) / X.std()  # Scale the predictors

# Coordinates for spatial modeling
coords = np.array([gdf_2015.geometry.x, gdf_2015.geometry.y]).T

# Response variables
n_trials = gdf_2015['total_count_shch'].values
positive = gdf_2015['positive_cases'].values

dtype('float64')

In [10]:
# Define mesh for SPDE approach
from pymc.gp.util import kmeans_inducing_points
n_knots = 20  # Number of knots for approximation
knot_locations = kmeans_inducing_points(coords, n_knots)

with pm.Model() as zib_model:
    # Priors for intercept and coefficients
    intercept = pm.Normal('intercept', mu=0, sigma=1)
    beta = pm.Normal('beta', mu=0, sigma=1, shape=X_scaled.shape[1])
    
    # Spatial random effect using SPDE
    # Length-scale and variance parameters
    ℓ = pm.Gamma('ℓ', alpha=2, beta=1)
    η = pm.HalfCauchy('η', beta=1)
    
    # Covariance function
    cov_func = η**2 * pm.gp.cov.Matern52(2, ℓ)
    
    # Approximate GP with inducing points
    gp = pm.gp.MarginalSparse(cov_func=cov_func, approx="FITC")
    
    # Spatial random effects
    f = gp.prior("f", X=knot_locations, Xu=knot_locations)
    f_pred = gp.conditional("f_pred", Xnew=coords)
    
    # Zero-inflation parameter
    psi = pm.Beta('psi', alpha=1, beta=5)  # Prior for zero-inflation
    
    # Linear predictor
    eta = intercept + pm.math.dot(X_scaled.values, beta) + f_pred
    p = pm.math.invlogit(eta)  # Convert to probability
    
    # Zero-inflated Binomial likelihood
    # First model the probability of structural zeros
    prob_zero = psi + (1 - psi) * (1 - p)**n_trials
    
    # Then model the count data with a mixture
    pm.ZeroInflatedBinomial('y', 
                           psi=psi,
                           n=n_trials,
                           p=p,
                           observed=positive)
    
    # Sample from the posterior
    trace = pm.sample(1000, tune=1000, cores=4, return_inferencedata=True)

TypeError: To use K-means initialization, please provide X as a type that can be cast to np.ndarray, instead of <class 'int'>